# CitizenTech · Monitor de Pavimento — Generative AI and Advanced Nets

**Global Solution 2026.2 · FIAP** · Agente generativo que transforma os dados dos módulos IoT e de Visão Computacional em decisões e respostas para a prefeitura e para o cidadão.

## 1. Problema, público e funcionalidade

**Problema.** Buracos nas vias causam acidentes e danos; a prefeitura só sabe deles quando alguém reclama, sem critério objetivo de prioridade. O CitizenTech coleta evidências automaticamente (sensores no veículo, câmeras, reportes) e as resume numa pontuação por trecho. Mas uma tabela de pontuações ainda exige que alguém a interprete, cruze com procedimentos e explique ao cidadão o que fazer.

**Público impactado.**
- **Equipe de zeladoria / subprefeitura:** precisa de um relatório de inspeção priorizado, com as evidências e uma rota sugerida, todos os dias, sem abrir o banco.
- **Cidadão:** precisa saber como reportar um buraco, como se proteger, quais são seus direitos, e quer registrar o que viu.

**Funcionalidade desenvolvida.** Um **agente de IA generativa** (LLM com ferramentas + RAG) que:
1. consulta o banco compartilhado e **explica o ranking** de prioridade em linguagem natural;
2. **gera o relatório de inspeção** com rota sugerida para a zeladoria;
3. **responde dúvidas do cidadão** com base nos documentos do projeto (RAG), citando a fonte;
4. **executa ações** na solução: registra reportes de cidadãos, analisa fotos com o detector YOLO e envia alertas MQTT ao dispositivo IoT.

**Benefício.** A prefeitura ganha tempo de triagem e decisões rastreáveis (cada número vem de uma consulta); o cidadão ganha orientação imediata e um canal que alimenta o mesmo sistema de prioridade.

## 2. Abordagem de IA escolhida e justificativa

| Abordagem estudada | Serve ao problema? | Decisão |
|---|---|---|
| **Transformers / Foundation Models** (LLM) | Sim: interpretar dados, gerar relatório, conversar | **Usada** — Claude (Anthropic) via LangChain |
| **Arquitetura agêntica** (ferramentas, ciclo pensar→agir→observar) | Sim: os números precisam vir do banco, e há ações a executar (registrar, alertar, analisar foto) | **Usada** — `langchain.agents.create_agent` com 9 ferramentas |
| **RAG** | Sim: procedimentos, direitos e limitações estão em documentos do projeto; evita alucinação e permite citar a fonte | **Usada** — FAISS + embeddings multilíngues locais |
| **Prompt engineering** | Sim: regras de dados, responsabilidade e formato do relatório | **Usada** — prompt de sistema por perfil (zeladoria / cidadão) |
| GANs, VAEs, Diffusion | Não: o problema não pede gerar imagens nem dados sintéticos | Não usadas (poderiam gerar imagens sintéticas de buracos para aumentar a base de visão: evolução futura) |

**Por que agente + RAG e não "um chatbot com um prompt grande"?** Porque (1) os dados mudam a cada leitura do ESP32 e a cada detecção — precisam ser consultados na hora, não colados no prompt; (2) parte da funcionalidade é **ação** (gravar no banco, publicar MQTT), que só um agente com ferramentas executa; (3) a resposta ao cidadão precisa ser **fiel a documentos** e citável, que é exatamente o que o RAG dá.

## 3. Arquitetura da solução generativa

```
                           ┌──────────────────────────────────────────────────────┐
  Zeladoria ──┐            │  Agente CitizenTech (LangChain create_agent)         │
              │  Gradio    │  LLM: Claude (claude-opus-5-5) · prompt por perfil    │
  Cidadão ────┼──────────▶ │  ciclo: pergunta → escolhe ferramenta → observa → ... │
  (foto)      │            └───────┬──────────────┬───────────────┬───────────────┘
              │                    │              │               │
              │           ┌────────▼───────┐ ┌────▼─────┐ ┌───────▼────────┐
              │           │ Ferramentas de │ │   RAG    │ │ Ferramentas de │
              │           │ consulta       │ │ FAISS +  │ │ ação           │
              │           │ ranking,       │ │ MiniLM   │ │ registrar      │
              │           │ trecho,        │ │ multi-   │ │ reporte,       │
              │           │ eventos,       │ │ língue   │ │ analisar foto  │
              │           │ dispositivos   │ │          │ │ (YOLO), alerta │
              │           └────────┬───────┘ └────┬─────┘ │ MQTT ao ESP32  │
              │                    │              │       └───┬────────┬───┘
              │           ┌────────▼──────────────▼───┐       │        │
              └──────────▶│  citizentech.db (SQLite)  │◀──────┘   ┌────▼──────┐
                          │  leituras/eventos IoT,    │           │  Broker   │
                          │  detecções de visão,      │           │  MQTT     │──▶ ESP32 (LED)
                          │  vw_prioridade_trechos    │           └───────────┘
                          └───────────▲───────────────┘
                   ESP32 → MQTT → receptor_mqtt.py          detector.py (YOLO11n)
```

| Componente | Papel | Arquivo |
|---|---|---|
| LLM (Claude Opus 5.5 via `langchain-anthropic`; trocável por variável de ambiente) | Interpreta a pergunta, decide ferramentas, redige | `agente.py` |
| Grafo do agente (`create_agent`, LangGraph) + memória por conversa (`InMemorySaver`) | Executa o ciclo de raciocínio/ação com histórico por `thread_id` | `agente.py` |
| 9 ferramentas (`@tool`) | Consultas ao banco, busca RAG, registro de reporte, análise de foto, alerta MQTT | `ferramentas.py` |
| RAG: 7 documentos `.md` + PDF do módulo IoT → 81 trechos → FAISS, embeddings `paraphrase-multilingual-MiniLM-L12-v2` | Conhecimento de procedimentos, direitos, limitações | `rag.py`, `base_conhecimento/` |
| Interface Gradio (abas Zeladoria, Cidadão, Integração) | Fluxo de interação e demonstração | `app_gradio.py` |
| Avaliação (12 cenários + juiz LLM) | Testes e qualidade | `avaliacao.py` |

In [ ]:
import os, sys, json, time, textwrap
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display, Image as IPImage

GENAI = Path.cwd().resolve()
if not (GENAI / "agente.py").exists():
    GENAI = (Path.cwd() / "citizentech" / "genai").resolve()
CITIZENTECH = GENAI.parent
sys.path.insert(0, str(GENAI)); sys.path.insert(0, str(CITIZENTECH)); sys.path.insert(0, str(CITIZENTECH / "visao"))
os.environ.setdefault("CITIZENTECH_MQTT_DRY_RUN", "1")   # no notebook, o alerta MQTT é simulado (sem rede)

import banco, rag, ferramentas, agente
MODELO = os.getenv("CITIZENTECH_LLM", agente.MODELO_PADRAO)
provedor = MODELO.split(":")[0]
chave = {"anthropic": "ANTHROPIC_API_KEY", "openai": "OPENAI_API_KEY", "google_genai": "GOOGLE_API_KEY", "groq": "GROQ_API_KEY"}.get(provedor)
ONLINE = bool(os.getenv(chave)) if chave else True
print(f"Modelo configurado: {MODELO} · chamadas ao LLM {'HABILITADAS' if ONLINE else 'DESABILITADAS (defina ' + str(chave) + ' para rodar as células do agente)'}")
import langchain, langchain_anthropic, gradio
print("langchain", langchain.__version__, "· langchain-anthropic", langchain_anthropic.__version__, "· gradio", gradio.__version__)

## 4. Base de conhecimento e RAG

Os documentos foram escritos para o projeto (procedimento de triagem, guia do cidadão, fórmula de prioridade, detalhes dos módulos, ODS, FAQ) e incluem o **relatório PDF do módulo IoT**. O pipeline divide cada documento em trechos de ~700 caracteres com sobreposição de 120, gera embeddings com um modelo multilíngue **local** (sem enviar documentos a terceiros) e indexa no FAISS. Na consulta, a pergunta vira embedding e os 4 trechos mais próximos são devolvidos ao agente com a fonte.

In [ ]:
docs = rag.carregar_documentos()
tabela = pd.DataFrame([{"fonte": d.metadata["fonte"], "título": d.metadata["titulo"], "caracteres": len(d.page_content)} for d in docs])
display(tabela.groupby("fonte").agg(títulos=("título", "count"), caracteres=("caracteres", "sum")))
pedacos = rag.dividir(docs)
print(f"{len(docs)} documentos → {len(pedacos)} trechos (chunk {rag.CHUNK}, sobreposição {rag.SOBREPOSICAO}) · embeddings: {rag.MODELO_EMBEDDINGS}")
indice = rag.construir_indice()

In [ ]:
perguntas = ["Como o cidadão reporta um buraco em São Paulo?",
             "Qual o prazo sugerido para vistoriar um trecho de prioridade alta?",
             "Por que a umidade aumenta a pontuação do trecho?",
             "Quais decisões exigem validação humana?"]
linhas = []
for q in perguntas:
    for doc, dist in rag.buscar(q, k=2):
        linhas.append({"pergunta": q, "distância": round(float(dist), 2), "fonte": doc.metadata["fonte"], "trecho": textwrap.shorten(doc.page_content.replace("\n", " "), 140)})
pd.set_option("display.max_colwidth", 150); pd.DataFrame(linhas)

## 5. Ferramentas do agente

Cada ferramenta é uma função Python decorada com `@tool`; o LangChain converte a assinatura e a docstring no esquema que o LLM vê. **Regra de projeto:** nenhum número sobre trechos vem da "memória" do modelo; tudo passa por uma ferramenta de consulta. As ferramentas de ação validam os dados (trecho existente, descrição mínima) antes de gravar.

In [ ]:
pd.DataFrame([{"ferramenta": t.name, "argumentos": ", ".join(t.args.keys()) or "—", "tipo": "ação" if t in ferramentas.FERRAMENTAS_ACAO else "consulta",
               "descrição (como o LLM vê)": textwrap.shorten(t.description.replace("\n", " "), 160)} for t in ferramentas.TODAS])

In [ ]:
print("consultar_ranking_prioridade →", ferramentas.consultar_ranking_prioridade.invoke({})[:600], "...\n")
print("detalhar_trecho('SIM9999') →", ferramentas.detalhar_trecho.invoke({"codigo_trecho": "SIM9999"}), "\n")
print("registrar_reporte_cidadao (descrição curta) →", ferramentas.registrar_reporte_cidadao.invoke({"codigo_trecho": "SIM0001", "descricao": "x"}), "\n")
print("buscar_base_conhecimento →", ferramentas.buscar_base_conhecimento.invoke({"pergunta": "o que a vistoria presencial deve confirmar?"})[:700], "...")

## 6. O agente: prompt, perfis e ciclo de execução

O prompt de sistema define identidade, **regras de dados** (nunca inventar; tudo via ferramenta), **regras de RAG** (buscar antes de responder sobre procedimentos; citar a fonte), **responsabilidade** (triagem ≠ vistoria; não é a prefeitura; confirmar antes de agir; sem dados pessoais) e o **formato do relatório**. Dois perfis ajustam o tom e o conjunto de ferramentas: o cidadão não pode enviar alertas ao ESP32.

In [ ]:
print(agente.PROMPT_BASE)
print("--- perfis ---")
for p, t in agente.PROMPT_PERFIL.items(): print(f"[{p}]{t}")
print("Ferramentas por perfil:", {p: [t.name for t in agente.ferramentas_do_perfil(p)] for p in ("zeladoria", "cidadao")})

### 6.1 Demonstração — perfil zeladoria
Cada chamada mostra o **rastro de ferramentas** (→ chamada, ← resultado) e a resposta final. As células seguintes só executam com a chave do modelo configurada; sem ela, o notebook explica como habilitar.

In [ ]:
def demo(perfil, pergunta, thread):
    if not ONLINE:
        print(f"[offline] defina {chave} (ou CITIZENTECH_LLM com outro provedor) para executar: {pergunta}"); return None
    ag = AGENTES.setdefault(perfil, agente.criar_agente(perfil))
    t0 = time.time(); resposta, passos = agente.perguntar(ag, pergunta, thread_id=thread)
    print(f"PERGUNTA ({perfil}): {pergunta}\n"); agente.imprimir_passos(passos)
    print(f"\n({time.time()-t0:.1f} s)"); display(Markdown(resposta)); return resposta, passos
AGENTES = {}
demo("zeladoria", "Quais trechos estão em prioridade alta hoje e que evidências sustentam isso?", "zel-1")

In [ ]:
demo("zeladoria", agente.PROMPT_RELATORIO, "zel-1")

### 6.2 Demonstração — perfil cidadão (RAG + ação)

In [ ]:
demo("cidadao", "Tem um buraco enorme na minha rua e a prefeitura não resolve. Como eu reporto? Tenho direito a alguma coisa se meu carro quebrar?", "cid-1")

In [ ]:
demo("cidadao", "Quero registrar aqui também: é na Rua Vergueiro perto do 1200, buraco grande com água acumulada. Pode usar o trecho SIM0001.", "cid-1")

## 7. Integração com os módulos IoT e Visão

### 7.1 Foto do cidadão → detector YOLO → banco → prioridade
A ferramenta `analisar_foto_buracos` chama o detector do módulo de Visão (`detector.py`) e, se o trecho for informado, grava as detecções confirmadas (confiança ≥ 0,5). Ela funciona mesmo sem o LLM; aqui mostramos a chamada direta e, em seguida, o agente usando-a dentro da conversa.

In [ ]:
from glob import glob
fotos = sorted(glob(str(CITIZENTECH / "visao" / "dataset" / "test" / "images" / "*.jpg")))
foto = fotos[5] if fotos else None
if foto:
    conn = banco.conectar(); banco.inicializar(conn)
    conn.execute("DELETE FROM deteccoes_visao WHERE arquivo = ?", (Path(foto).name,)); conn.commit()
    antes = banco.prioridade_do_trecho(conn, "SIM0003")
    res = json.loads(ferramentas.analisar_foto_buracos.invoke({"caminho_imagem": foto, "codigo_trecho": "SIM0003"}))
    depois = banco.prioridade_do_trecho(conn, "SIM0003")
    if "erro" in res:
        print("Ferramenta respondeu com erro (treine o detector primeiro):", res["erro"])
    else:
        print(json.dumps({k: v for k, v in res.items() if k != "deteccoes"}, ensure_ascii=False, indent=1))
        display(pd.DataFrame(res["deteccoes"]))
        print(f"Trecho {depois['nome']}: pontuação {antes['pontuacao']} ({antes['nivel']}) → {depois['pontuacao']} ({depois['nivel']})")
        display(IPImage(filename=res["imagem_anotada"], width=480))
else:
    print("Rode primeiro citizentech/visao/preparar_dados.py para ter fotos de teste.")

In [ ]:
if foto: demo("cidadao", f"Enviei uma foto da via em {foto}. Analise com o detector de buracos e me diga o que encontrou, sem gravar no banco.", "cid-2")

### 7.2 Agente → comando MQTT → ESP32 (LED vermelho piscando)
O mesmo tópico de comando que o receptor IoT usa (`<base>/<device>/comando`). No notebook o envio é simulado (`CITIZENTECH_MQTT_DRY_RUN=1`); no app, com o ESP32 ligado, o LED pisca por 5 s e o dispositivo responde com o evento `alerta_recebido`.

In [ ]:
print(ferramentas.enviar_alerta_esp32.invoke({"device_id": "esp32-01", "motivo": "trecho crítico: Av. Lins de Vasconcelos"}))
demo("zeladoria", "O veículo esp32-01 está entrando no trecho de maior prioridade. Se ele estiver em nível alta, envie o alerta ao dispositivo e me confirme.", "zel-2")

## 8. Testes, avaliação e qualidade dos resultados

`avaliacao.py` executa **12 cenários** cobrindo consulta ao banco, geração de relatório, RAG, ação, alucinação (trecho inexistente, pergunta fora da base), segurança (pedido de dados pessoais) e raciocínio sobre os dados. Para cada caso são verificados automaticamente: ferramentas esperadas chamadas, termos obrigatórios/proibidos, admissão de desconhecimento, citação de fonte. Um segundo LLM atua como **juiz** (coerência, relevância, utilidade, fidelidade, 1–5). Os resultados completos ficam em `evidencias/avaliacao_resultados.md`.

In [ ]:
import avaliacao
display(pd.DataFrame([{"#": c["id"], "categoria": c["categoria"], "perfil": c["perfil"], "pergunta": textwrap.shorten(c["pergunta"], 90),
                       "ferramentas esperadas": ", ".join(c["espera_ferramentas"]) or "—", "verificações extras": ", ".join(k for k in ("deve_admitir", "deve_citar_fonte") if c.get(k)) or "—"}
                      for c in avaliacao.CASOS]))

In [ ]:
EVID = GENAI / "evidencias"
if ONLINE:
    agentes_av = {p: agente.criar_agente(p) for p in ("zeladoria", "cidadao")}
    juiz = agente.criar_modelo()
    resultados = [avaliacao.avaliar_caso(c, agentes_av, juiz) for c in avaliacao.CASOS]
    EVID.mkdir(exist_ok=True)
    (EVID / "avaliacao_resultados.json").write_text(json.dumps(resultados, ensure_ascii=False, indent=1, default=str))
    avaliacao.gerar_relatorio(resultados, EVID / "avaliacao_resultados.md")
elif (EVID / "avaliacao_resultados.json").exists():
    resultados = json.loads((EVID / "avaliacao_resultados.json").read_text()); print("(resultados carregados de uma execução anterior)")
else:
    resultados = []; print("[offline] sem resultados de avaliação ainda; execute avaliacao.py com a chave configurada.")
if resultados:
    tab = pd.DataFrame([{"#": r["id"], "categoria": r["categoria"], "ferramentas": ", ".join(r["ferramentas_chamadas"]) or "—",
                         "aprovado": "sim" if r["aprovado"] else "NÃO", **{f"juiz {k}": (r.get("notas_juiz") or {}).get(k, "-") for k in ("coerencia", "relevancia", "utilidade", "fidelidade")},
                         "s": r["duracao_s"]} for r in resultados])
    display(tab)
    print(f"Aprovados nas verificações automáticas: {sum(r['aprovado'] for r in resultados)}/{len(resultados)}")

**Como ler os resultados.** As verificações automáticas são objetivas (ferramenta certa chamada, termo presente, fonte citada, desconhecimento admitido). As notas do juiz são subjetivas e servem para comparar versões do prompt. Casos típicos de falha observados durante o desenvolvimento e como foram tratados:
- *Trecho inexistente* (caso 4): sem a regra "nunca invente", o modelo tende a responder com uma pontuação plausível; com a ferramenta devolvendo o erro + lista de códigos válidos, ele admite e oferece as opções.
- *Pergunta fora da base* (caso 12): o RAG devolve trechos pouco relacionados; o prompt instrui a dizer que a informação não está na base em vez de improvisar.
- *Dados pessoais* (caso 9): recusa e explica que o protótipo não guarda nem fornece dados pessoais.
- *Ações* (caso 8): o agente só chama `registrar_reporte_cidadao` quando o trecho e a descrição estão claros; descrições curtas são rejeitadas pela própria ferramenta.

## 9. Limitações, segurança e responsabilidade

| Risco | Como se manifesta | Mitigação adotada | O que ainda exige validação humana |
|---|---|---|---|
| **Alucinação de dados** | números ou trechos inventados | toda informação sobre trechos vem de ferramentas; erro explícito para códigos inválidos; prompt "nunca invente" | qualquer decisão de reparo baseada no ranking |
| **Resposta fora da base / desatualizada** | orientação jurídica ou procedimento errado | RAG com citação de fonte; instrução para admitir desconhecimento e indicar canal oficial (SP156) | pareceres jurídicos e comunicação oficial ao cidadão |
| **Ação indevida** | reporte registrado sem querer; alerta enviado ao veículo por engano | confirmação antes de agir; validação na ferramenta; perfil cidadão sem acesso ao alerta MQTT; modo simulado no notebook | envio de alertas e qualquer comando ao dispositivo em operação real |
| **Privacidade** | coleta de nome, CPF, telefone, localização precisa | o prompt proíbe pedir/registrar dados pessoais; reporte guarda só trecho + descrição; fotos processadas localmente | política de retenção e LGPD numa implantação real |
| **Viés** | priorizar bairros com mais reportes digitais (quem reporta mais) | a pontuação combina sensores automáticos + visão + reportes, reduzindo o peso do "volume de reclamação"; pesos documentados e ajustáveis | calibração dos pesos com vistorias reais; auditoria da distribuição por região |
| **Dependência do provedor / custo** | indisponibilidade da API, custo por token | modelo trocável por variável de ambiente (`CITIZENTECH_LLM`); ferramentas e RAG funcionam offline | — |
| **Segurança do canal MQTT** | broker público sem autenticação | fora do escopo do protótipo; em produção, broker próprio com TLS e ACL | — |
| **Confiabilidade do detector** | falsos positivos entram na pontuação | só detecções ≥ 0,5 contam; vídeo grava um registro por buraco rastreado | confirmação da gravidade na vistoria |

**Decisões que continuam humanas:** confirmar a gravidade de um buraco, decidir reparo e sinalização, responder a pedidos de indenização e qualquer comunicação oficial.

## 10. Demonstração da aplicação e integração com a solução CitizenTech

**Fluxo ponta a ponta demonstrado no vídeo:** ESP32 mede um buraco → MQTT → receptor grava no banco → trecho sobe para prioridade alta → detector de visão confirma buracos no vídeo da rua e grava as detecções → zeladoria pede o relatório ao agente, que consulta o ranking, detalha os trechos, monta a rota e envia o alerta ao veículo → cidadão pergunta como reportar e registra o buraco pelo mesmo agente.

Para abrir a interface:
```bash
export ANTHROPIC_API_KEY=sk-ant-...        # ou CITIZENTECH_LLM=openai:gpt-5 + OPENAI_API_KEY
.venv/bin/python citizentech/genai/app_gradio.py     # http://localhost:7860
```

In [ ]:
# Para abrir a interface dentro do notebook, descomente:
# import app_gradio; app_gradio.app.launch(share=False, inline=True, height=720)
print("Abas da interface:", ["Zeladoria", "Cidadão", "Integração (banco compartilhado)"])

**Evolução futura.** (1) Memória persistente das conversas (checkpointer em SQLite) e histórico de relatórios; (2) busca híbrida (BM25 + vetorial) e re-ranking para a base crescer; (3) geração de imagens sintéticas de buracos (Diffusion) para aumentar a base do detector; (4) *guardrails* com classificador de entrada para detectar pedidos fora do escopo; (5) avaliação contínua com casos reais coletados da interface.